# Record Renting System
#
# Verifications
Program: Written by F430059 the purpose of this cell is to load the external python files given to us, as use them to verify both the User's ID and the record ID, ensuring the user is registered on the system and the record ID is also present within the system

Function: ValidateSUB is just to validate that a user is actually subscribed and able to rent / return records on this system. If they arent verified, they simply wont be able to rent or return records. A customer will give the system manager these details at a till for example.

In [68]:
import subscriptionManager as subm
import feedbackManager as feedm
import datetime as dt
from random import randint 

subscriptions = subm.load_subscriptions()
feedback_list = feedm.load_feedback()
def ValidateSUB(userID):
    #Use this to check if the user is subscribed
    global ValidatedUserSUB
    ValidatedUserSUB = False
    if len(userID) != 4:
        print("Your userID can only be 4 characters long. Try again!")
    else:
        if subm.check_subscription(userID, subscriptions) == True:
            print("User found!")
            ValidatedUserSUB = True
        else:
            print("UserID \'{}\' wasn\'t recognised. Try again!".format(userID))
                
def ValidateRID(RecordID):
    global ValidatedRecID
    ValidatedRecID = False
    if len(RecordID) != 5:
        print("Your RecordID must be 5 characters long. Try again!")
    else:
        f = open("Music_info.txt", "r").readlines()
        while True:
            for line in f:
                line = line.split(",")
                if line[0] == RecordID:
                    print("RecordID Valid")
                    ValidatedRecID = True
                    return
        print("RecordID \'{}\' wasn\'t found. Try again!".format(RecordID))

# musicRent

"A Python module containing functions that prompt the store manager for the customer's
ID and the ID of the music-record(s) they wish to rent. After performing validity checks and the
functionality described earlier, the program should return a message indicating whether the music has
been rented successfully."

Program: Written by F430059. The purpose of this cell is to enable users to rent records and consequently, the system will update the backend database "rentals.txt" to say this record is actively being rented by a new user. 

Function: The function musicRent is responsible for all new rentals, it will automatically delete pre-exisiting data which in a real life situation would meet help a company to meet GDPR laws, as they remove old data that they dont need anymore. This function also takes the real date when updating an old date in the file.

In [69]:
def musicRent(): 
    userID = userID_input.value
    RecordID = recordID_input.value
    if ValidatedUserSUB == False or ValidatedRecID == False:
        return
    updatedRentals = []
    date = str(dt.datetime.now().date())
    dateAdj = date.split("-")
    count = 0
    m = open("Rental.txt", "r").readlines()
    for line in m:
        tempStore = line.split(",")
        if line.strip() != "" and tempStore[2].strip() == "" and tempStore[3].strip() == userID:
            count += 1 #Used to count the number of records the User is actively renting 

    q = open("Subscription_Info.txt", "r").readlines()
    for line in q:
        tempStore = line.split(",")
        if tempStore[0] == userID:
            rentalLimit = subm.get_rental_limit(tempStore[1])
        
    dateAdj2 = [int(dateAdj[0]), int(dateAdj[1]), int(dateAdj[2])]
    year, month, day = dateAdj[0], dateAdj[1], dateAdj[2]
    if ValidatedRecID == True and ValidatedUserSUB == True and count < rentalLimit: 
        f = open("Rental.txt", "r").readlines()
        for line in f:
            tempSearch = line.split(",") 
            if tempSearch[0] == RecordID: 
                if tempSearch[2].strip() == "": #Might be unnecessary as last line should suffice?
                    print("The record ID \'{}\' is actively being rented! Try again another time.".format(RecordID))
                    updatedRentals.append(line)
                else:
                    ReturnDate = tempSearch[2].split("-")
                    ReturnDate2 = [int(ReturnDate[0]), int(ReturnDate[1]), int(ReturnDate[2])]
                    b1 = dt.date(dateAdj2[0], dateAdj2[1], dateAdj2[2]) #Putting the date from the file in a comparable format to compare to the real time
                    b2 = dt.date(ReturnDate2[0], ReturnDate2[1], ReturnDate2[2])
                    if b1 > b2:
                        updatedRentals.append("{},{}, ,{}\n".format(RecordID, date, userID)) 
                    else:
                        print("The record ID \'{}\' is actively being rented! Try again another time.".format(RecordID))
                        updatedRentals.append(line)
            else:
                updatedRentals.append(line)
        with open("Rental.txt", "w") as q:
            q.writelines(updatedRentals)
        print("Successfully rented the recordID: {} as of: {}".format(RecordID, date))
    else:
        print("The users has reached their rental limit ({}). Return one before renting again!".format(rentalLimit))

    userID_input.layout.display = 'none'
    ValidateSUB_button.layout.display = 'none'
    recordID_input.layout.display = 'none'
    ValidateRID_button.layout.display = 'none'

# musicSearch
"A Python module containing functions that allow the store manager to input search
terms as strings and return the output"

Program: Written by F430059. The purpose of this cell is to skim the database for certain strings of data, it should be used by entering a string and pressing search. If the string is found, all matches will be shown.

Function: The function musicSearch is for taking an input and scanning every single piece of data within MusicInfo, to see what in the database matches the search. If something does match, all of its associated data will also be displayed


In [70]:
def musicSearch():
    foundMatch = False
    f = open("Music_Info.txt", 'r').readlines()
    search = musicSearch_input.value.lower().strip()
    for row in f:
        x = row.split(',')
        for i in range(6):
            if x[i-1].lower().strip() == search:
                foundMatch = True
                print("Found a record with:\n\nArtist Name: {}\nTitle: {}\nMedium: {}\nGenre: {}\n".format(x[1], x[2], x[3], x[4])) #Make more pretty
    if foundMatch == False:
        print("No matches were found in our database!\n\nEnsure you are searching for music based on artist, \ntitle (_ for instead of spaces), medium (CD, vinyl or tape), \ngenre (e.g. pop, classical, jazz, hip_hop, rap, kwaito)")

# musicReturn
"A Python module containing functions that prompt the store manager for the ID of the
music-record(s) they wish to return and collect feedback if applicable."

Program: Written by F430059. The purpose of this cell is to return records and make them available again in the database

Function: The function musicReturn begins by verifying the userID and recordID. It also checks to ensure the person returning the record is the same person who last rented it. After all checks, it will collect feedback, which is then managed by feedback manager

In [71]:
def musicReturn():
    userID_input.layout.display = 'none'
    ValidateSUB_button.layout.display = 'none'
    recordID_input.layout.display = 'none'
    ValidateRID_button.layout.display = 'none'
    StarRating_input.layout.display = 'none'
    OptFeedback_fix.layout.display = 'none'
    SubmitBtn.layout.display = 'none'
    userID = userID_input.value
    RecordID = recordID_input.value
    updatedRentals = []
    date = str(dt.datetime.now().date())
    ValidateRID(RecordID)
    ValidateSUB(userID)
    if ValidatedRecID == False:
        return
    if ValidatedUserSUB == False:
        return
    f = open("Rental.txt", "r").readlines()
    for line in f:
        tempSearch = line.split(",") #Temporarily hold a line from rental.txt
        if tempSearch[0] == RecordID:
            if tempSearch[2].strip() == "" and userID.strip() == tempSearch[3].strip():
                updatedRentals.append("{},{},{},{}\n".format(RecordID,tempSearch[1],date,userID))
            else:
                print("This record isn\'t returnable. Likely because a different user rented initially!")
                return
        else:
            updatedRentals.append(line)
    with open("Rental.txt", "w") as q:
        q.writelines(updatedRentals)
    StarRating = StarRating_input.value
    OptFeedback = OptFeedback_input.value.strip() 
    if OptFeedback_input.value.strip() == "": 
        OptFeedback = "N/A"
    feedm.add_feedback(RecordID,StarRating,OptFeedback,"Music_Feedback.txt")
    output.clear_output()
    print("Successfully returned the recordID: {} as of: {}".format(RecordID, date))


# InventoryPruning
"A Python module containing functions used to identify music-records for potential
removal based on rental frequency. Before any pruning action, the module should provide suggestions
and aid the decision-making process by offering visualisations."

Program: Written by F430059. The following two cells are responsible for pruning the inventory of un-wanted records, which helps to save space

Function: the function invPrune is responsible for the deletion of records from rentals.txt. Whatever records should be deleted is decided by the store manager who will have a visual representation of the least desired records. 

In [72]:
def invPrune():
    toPrune = invPrune_input.value
    f = open("Rental.txt", "r").readlines()
    updatedRentals = []
    for line in f:
        tempStore = line.split(",")
        if tempStore[0] != "\n":
            if tempStore[2].strip() != "" and tempStore[0] != "RecordID":
                if tempStore[0] == toPrune:
                    print("Successfully pruned the record:", tempStore[0])
                else:
                    updatedRentals.append(line)
            else:
                updatedRentals.append(line)
            
    with open("Rental.txt", "w") as f:
        f.writelines(updatedRentals)


# Inventory Pruning Bar-Graph
Program: Written by F430059 - Inv pruning visual representation

Function: display_bar_chart() is necessary for the visual representation of which records arent desired. Its a bar chart that is dynamic and updated as soon as a record is pruned to help store managers quickly identify which records arent being rented often. The way it does this is by identifying the current date, and comparing this to the date a record was last returned, consequently this function will pay no attention to records that are actively being rented

In [73]:
def display_bar_chart():
    global recordNames
    global DaysSinceRented
    recordNames , DaysSinceRented = [] , []
    date = dt.datetime.now().date()
    f = open("Rental.txt", "r").readlines()
    
    for line in f:
        tempStore = line.split(",")
        if tempStore[0] != "\n":
            if tempStore[2].strip() != "" and tempStore[0] != "RecordID":
                rentalDate = dt.datetime.strptime(tempStore[2], "%Y-%m-%d").date()
                if (date-rentalDate).days >= 30:
                    recordNames.append(tempStore[0])
                    DaysSinceRented.append((date-rentalDate).days)

    plt.figure(figsize=(max(10, len(recordNames) * 0.5), 6))
    plt.bar(recordNames, DaysSinceRented, color="orange")
    plt.xlabel('Names of records')
    plt.ylabel('Days since last rented')
    plt.title('Bar Chart of how many days since record was last rented (Only shows beyond 30 days)!')
    plt.tight_layout()
    plt.show()

# Menu.ipynb
"The main program that provides the required menu options to the store manager."

Program: Written by F430059 - This cell is for the store manager to interact with the program, they'll be faced with human readable buttons and interactive widgets.

Function: 
musicRent_Clicked() - A button used to prompt the user to rent 

musicReturn_clicked() - A button used to prompt the user
          
onValidateSUB_clicked() - A button used to verify the subscription of the UserID that has been inputted. 

onValidateRID_clicked() - A button used to verify the record id thats inputted to the text field
          
onSubmitBtn_clicked() - A button used to call the functions musicReturn, validate record ID and validate subscription

musicSearch_clicked() - A button used to prompt the user

invPrune_clicked() - A button used to display the bar chart outlining how many days since each record was rented. Due to size limits its               been set to only display records where its not been rented for beyond 30 days

pruneNow_clicked() - A button used to delete a record, likely due to the fact that its not rented often

In [74]:
from ipywidgets import interact
import ipywidgets as widgets
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from ipywidgets import Layout
from IPython.display import clear_output

ValidatedUserSUB = False
ValidatedRecID = False
JustCalled = ""

StarRating_input = widgets.Text(description="Enter a Star rating (1-5)*:", layout=Layout(width='200px'))
OptFeedback_title = widgets.Label(value="Optional Feedback:")
OptFeedback_input = widgets.Text(layout=Layout(width='300px'))
OptFeedback_fix = widgets.VBox([OptFeedback_title, OptFeedback_input])
musicRent_button = widgets.Button(description="Rent Record") 
userID_input = widgets.Text(description="User ID*:", layout=Layout(width='100px'))
ValidateSUB_button = widgets.Button(description="Validate Subscription")
recordID_input = widgets.Text(description="Record ID*:", layout=Layout(width='100px'))
ValidateRID_button = widgets.Button(description="Validate Record ID")
musicReturn_button = widgets.Button(description="Return Record") 
SubmitBtn = widgets.Button(description="Submit")
musicSearch_button = widgets.Button(description="⌕", layout=Layout(width='30px'))
musicSearch_input = widgets.Text(placeholder="Search...", layout=Layout(width='180px'))
invPrune_button = widgets.Button(description="Inventory Prune")
invPrune_input = widgets.Text(description="Which record would you like to prune:", layout=Layout(width='100px'))
pruneNow_button = widgets.Button(description="Prune Now!") 

output = widgets.Output()

userID_input.layout.display = 'none'
invPrune_input.layout.display = 'none'
ValidateSUB_button.layout.display = 'none'
recordID_input.layout.display = 'none'
ValidateRID_button.layout.display = 'none'
StarRating_input.layout.display = 'none'
OptFeedback_fix.layout.display = 'none'
SubmitBtn.layout.display = 'none'
pruneNow_button.layout.display = "none"
musicSearch_input.layout.display = 'none'

def musicRent_clicked(b):
    output.clear_output()
    global ValidatedUserSUB
    global ValidatedRecID
    global JustCalled
    JustCalled = "rent"
    ValidatedUserSUB = False
    ValidatedRecID = False
    userID_input.layout.display = 'block'
    ValidateSUB_button.layout.display = 'block'
    recordID_input.layout.display = 'block'
    ValidateRID_button.layout.display = 'block'
    StarRating_input.layout.display = 'none'
    OptFeedback_fix.layout.display = 'none'
    SubmitBtn.layout.display = 'none'
    invPrune_input.layout.display = 'none'
    musicSearch_input.layout.display = 'none'
    pruneNow_button.layout.display = "none"
    
    with output:
        output.clear_output()
        print("Once a valid recordID and userID are entered, the record will automatically be rented!")

def musicReturn_clicked(b):
    output.clear_output()
    global ValidatedUserSUB
    global ValidatedRecID
    global JustCalled
    JustCalled = "return"
    ValidatedUserSUB = False
    ValidatedRecID = False
    StarRating_input.layout.display = 'block'
    OptFeedback_fix.layout.display = 'block'
    userID_input.layout.display = 'block'
    ValidateSUB_button.layout.display = 'none'
    recordID_input.layout.display = 'block'
    ValidateRID_button.layout.display = 'none'
    SubmitBtn.layout.display = 'block'
    invPrune_input.layout.display = 'none'
    musicSearch_input.layout.display = 'none'
    pruneNow_button.layout.display = "none"
    
    with output:
        output.clear_output()
        print("A userID, record ID and Star rating are all required beforee you press submit!")

def onValidateSUB_clicked(b):
    global ValidatedUserSUB
    with output:
        output.clear_output()
        ValidateSUB(userID_input.value)
        if ValidatedUserSUB == True and ValidatedRecID == True:
            if JustCalled == "rent":
                musicRent()
            else:
                musicReturn()
        
def onValidateRID_clicked(b):
    global ValidatedRecID
    with output:
        output.clear_output()
        ValidateRID(recordID_input.value)
        if ValidatedUserSUB == True and ValidatedRecID == True:
            if JustCalled == "rent":
                musicRent()
            else:
                musicReturn()

def onSubmitBtn_clicked(b):
    global ValidatedRecID
    global ValidatedUserSUB
    with output:
        output.clear_output()
        ValidateSUB(userID_input.value)
        ValidateRID(recordID_input.value)
        if ValidatedUserSUB == True and ValidatedRecID == True:
            musicReturn()

def musicSearch_clicked(b):
    output.clear_output()
    musicSearch_input.layout.display = 'block'
    userID_input.layout.display = 'none'
    recordID_input.layout.display = 'none'
    StarRating_input.layout.display = 'none'
    OptFeedback_fix.layout.display = 'none'
    ValidateSUB_button.layout.display = 'none'
    ValidateRID_button.layout.display = 'none'
    SubmitBtn.layout.display = 'none'
    invPrune_input.layout.display = 'none'
    musicSearch_input.layout.display = 'block'
    pruneNow_button.layout.display = "none"
    with output:
        output.clear_output()
        print("Search to find music based on artist, title, medium or genre. Press search again to search!")
        if musicSearch_input.value.strip() != "":
            musicSearch()

def invPrune_clicked(b):
    output.clear_output()
    musicSearch_input.layout.display = 'none'
    userID_input.layout.display = 'none'
    recordID_input.layout.display = 'none'
    StarRating_input.layout.display = 'none'
    OptFeedback_fix.layout.display = 'none'
    ValidateSUB_button.layout.display = 'none'
    ValidateRID_button.layout.display = 'none'
    SubmitBtn.layout.display = 'none'
    musicSearch_input.layout.display = 'none'
    pruneNow_button.layout.display = "block"
    with output:
        output.clear_output()
        display_bar_chart()
        invPrune_input.layout.display = 'block'

def pruneNow_clicked(b):
    with output:
        if invPrune_input.value.strip() != "":
            invPrune()
            output.clear_output()
            display_bar_chart()
            
musicRent_button.on_click(musicRent_clicked)
musicReturn_button.on_click(musicReturn_clicked)
musicSearch_button.on_click(musicSearch_clicked)
ValidateSUB_button.on_click(onValidateSUB_clicked)
ValidateRID_button.on_click(onValidateRID_clicked)
pruneNow_button.on_click(pruneNow_clicked)
SubmitBtn.on_click(onSubmitBtn_clicked)
invPrune_button.on_click(invPrune_clicked)


buttons = widgets.HBox([
    musicRent_button,
    musicReturn_button,
    invPrune_button,
    musicSearch_input,
    musicSearch_button
])
inputs = widgets.VBox([
    userID_input,
    ValidateSUB_button,
    recordID_input,
    StarRating_input,
    OptFeedback_fix,
    ValidateRID_button,
    SubmitBtn,
    invPrune_input,
    pruneNow_button
])

layout = widgets.VBox([
    buttons,
    inputs,
    output
])

layout